In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from keras.layers import Dense
from keras import Sequential
from lazypredict.Supervised import LazyRegressor
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [3]:
df = pd.read_csv('bayut_listings_ready.csv')
df.head()

,Price,Property Type,Number Of Bedrooms,Number Of Bathrooms,Area,Location
0,0.00,Floor,0.36,0.40,0.00,"Al Khaleej, East Riyadh, Riyadh"
1,0.00,Apartment,0.14,0.20,0.00,"Al Sakb, Madina"
2,0.00,Apartment,0.14,0.20,0.00,"Al Sakb, Madina"
3,0.00,Apartment,0.14,0.20,0.00,"Al Sakb, Madina"
4,0.00,Apartment,0.36,0.40,0.00,"Al Rabwa, North Jeddah, Jeddah"


In [4]:
X = df.drop(columns='Price', axis=1)
y = df['Price']
col = X.drop(columns='Property Type').columns

In [5]:
df.head(1)

,Price,Property Type,Number Of Bedrooms,Number Of Bathrooms,Area,Location
0,0.00,Floor,0.36,0.40,0.00,"Al Khaleej, East Riyadh, Riyadh"


In [6]:
targetenc = TargetEncoder(smooth='auto')
X[col] = targetenc.fit_transform(X[col],y)


In [7]:
X.head()

,Property Type,Number Of Bedrooms,Number Of Bathrooms,Area,Location
0,Floor,0.00,0.00,0.00,0.00
1,Apartment,0.00,0.00,0.00,0.00
2,Apartment,0.00,0.00,0.00,0.00
3,Apartment,0.00,0.00,0.00,0.00
4,Apartment,0.00,0.00,0.00,0.00


In [8]:
X_encoded = pd.get_dummies(X,columns=['Property Type'],dtype=int)
X_encoded.head()

,Number Of Bedrooms,Number Of Bathrooms,Area,Location,Property Type_Apartment,Property Type_Building,Property Type_Floor,Property Type_Residential Building,Property Type_Rest House,Property Type_Villa
0,0.00,0.00,0.00,0.00,0,0,1,0,0,0
1,0.00,0.00,0.00,0.00,1,0,0,0,0,0
2,0.00,0.00,0.00,0.00,1,0,0,0,0,0
3,0.00,0.00,0.00,0.00,1,0,0,0,0,0
4,0.00,0.00,0.00,0.00,1,0,0,0,0,0


In [9]:
x_train,x_test,y_train,y_test = train_test_split(X_encoded,y,test_size=0.2,random_state=32)

In [10]:
# model_linear = LinearRegression().fit(x_train,y_train)
# model_bay = BayesianRidge().fit(x_train,y_train)
# model_lasso = Lasso().fit(x_train,y_train)
# print(f'model performance on training data: {model_linear.score(x_train,y_train)} \nmodel performance on testing data: {model_linear.score(x_test,y_test)}')
# print(f'model_bay performance on training data: {model_bay.score(x_train,y_train)} \nmodel_bay performance on testing data: {model_bay.score(x_test,y_test)}')
# print(f'model_lasso performance on training data: {model_lasso.score(x_train,y_train)} \nmodel_lasso performance on testing data: {model_lasso.score(x_test,y_test)}')

In [11]:
# model = tf.keras.Sequential([Dense(units=25,activation='relu'),
#                              Dense(units=15,activation='relu'),
#                              Dense(units=1,activation= tf.keras.activations.extratreesregressor)])
# model.compile(loss=tf.keras.losses.MeanSquaredError(
#     reduction='sum_over_batch_size',
#     name='mean_squared_error'),
#     metrics=[tf.keras.metrics.MeanSquaredError()]
# )
# history = model.fit(
#     x_train,
#     y_train,
#     batch_size=64,
#     epochs=100,
#     # We pass some validation for
#     # monitoring validation loss and metrics
#     # at the end of each epoch
   
# )

In [12]:
# print(history.history) # this will print a dictionary object, now you need to grab the metrics / score you're looking for


In [13]:
# loss, accuracy = model.evaluate(x_train, x_train, batch_size=1000)
# print(f"Test Loss: {loss:.4f}")
# print(f"Test Accuracy: {accuracy:.4f}")

In [14]:
# reg = LazyRegressor(verbose=0,ignore_warnings=False, custom_metric=None )
# models,predictions = reg.fit(x_train, x_test, y_train, y_test)
# models.head()

In [15]:
neigh = KNeighborsRegressor(n_neighbors=1)
neigh.fit(x_train,y_train)
neigh.score(x_train,y_train)

0.9862270328517695

In [16]:
neigh.score(x_test,y_test)

0.42008625415351397

In [ ]:
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

pipeline = Pipeline([
    # ('scalar', StandardScaler()),
     ('poly', PolynomialFeatures(degree=2)),
    ('model', RandomForestRegressor(random_state=42))
])

grid = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, verbose=1)
grid.fit(x_train, y_train)

print("Best parameters:", grid.best_params_)
print("Train score:", grid.score(x_train, y_train))
print("Test score:", grid.score(x_test, y_test))


Fitting 5 folds for each of 90 candidates, totalling 450 fits
Best parameters: {'model__max_depth': None, 'model__min_samples_leaf': 4, 'model__min_samples_split': 10, 'model__n_estimators': 200}
Train score: 0.6618129576901219
Test score: 0.594946634313084


In [ ]:
param_grid = {
    'model__n_estimators': [200],
    'model__max_depth': [32],
    'model__min_samples_split': [9],
    'model__min_samples_leaf': [4]
}

pipeline = Pipeline([
    # ('scalar', StandardScaler()),
     ('poly', PolynomialFeatures(degree=2)),
    ('model', RandomForestRegressor(random_state=42))
])

grid = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, verbose=1,scoring='r2')
grid.fit(x_train, y_train)

print("Best parameters:", grid.best_params_)
print("Train score:", grid.score(x_train, y_train))
print("Test score:", grid.score(x_test, y_test))

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Best parameters: {'model__max_depth': 32, 'model__min_samples_leaf': 4, 'model__min_samples_split': 9, 'model__n_estimators': 200}
Train score: 0.6614792046354504
Test score: 0.5928923755884563


In [18]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_neighbors': list(range(1, 30))
}

pipeline = Pipeline([
    ('scalar', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2)),
    ('model', KNeighborsRegressor())
])

grid = GridSearchCV(pipeline, param_grid, cv=5)
grid.fit(x_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best cross-validation score:", grid.best_score_)
print("Test score:", grid.score(x_test, y_test))


Best parameters: {'model__n_neighbors': 29}
Best cross-validation score: 0.47219840631784715
Test score: 0.49319897015490743


In [19]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(pipeline, x_train, y_train, cv=5)
print("Cross-validation scores:", scores)
print("Average CV score:", scores.mean())

Cross-validation scores: [0.29233866 0.44533923 0.40247835 0.2938388  0.5296019 ]
Average CV score: 0.39271938619700986
